# Quark vs Gluon Jet Classification

This notebook builds a complete machine learning pipeline to classify particle jets as **quark-initiated** or **gluon-initiated** using the **EnergyFlow quark–gluon jets dataset** generated with **PYTHIA**.

It includes:

- feature engineering from particle-level jet data
- baseline models: **Random Forest** and **XGBoost**
- deep learning with **CNN jet images**
- evaluation using **accuracy**, **ROC**, **AUC**, and **confusion matrices**
- physics-motivated visualizations

## 1. Install dependencies

In [ ]:
!pip -q install energyflow xgboost scikit-learn matplotlib seaborn tensorflow pandas

## 2. Imports and setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, roc_auc_score, roc_curve,
    confusion_matrix, classification_report
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

from energyflow.datasets import qg_jets

sns.set_style("whitegrid")
sns.set_context("talk")

np.random.seed(42)
tf.random.set_seed(42)

SAVE_DIR = "qg_results"
os.makedirs(SAVE_DIR, exist_ok=True)

print("Results will be saved to:", SAVE_DIR)

## 3. Load dataset

In [ ]:
X, y = qg_jets.load(
    num_data=120000,
    pad=True,
    ncol=4,
    generator='pythia',
    with_bc=False
)

print("X shape:", X.shape)   # (N, max_particles, 4)
print("y shape:", y.shape)
print("Labels:", np.unique(y))

## 4. Dataset notes

Each particle constituent is represented as:

- `pt`  : transverse momentum
- `y`   : rapidity
- `phi` : azimuthal angle
- `pid` : particle ID

In [ ]:
print("First jet, first 10 particles:\n")
print(X[0, :10])

## 5. Feature engineering

In [ ]:
def safe_delta_phi(phi1, phi2):
    dphi = phi1 - phi2
    return (dphi + np.pi) % (2 * np.pi) - np.pi


def compute_jet_features(X):
    pt  = X[:, :, 0]
    y   = X[:, :, 1]
    phi = X[:, :, 2]

    mask = pt > 0
    n_particles = mask.sum(axis=1)
    jet_pt = pt.sum(axis=1)

    px = pt * np.cos(phi)
    py = pt * np.sin(phi)
    pz = pt * np.sinh(y)
    E  = pt * np.cosh(y)

    jet_px = px.sum(axis=1)
    jet_py = py.sum(axis=1)
    jet_pz = pz.sum(axis=1)
    jet_E  = E.sum(axis=1)

    jet_mass2 = jet_E**2 - jet_px**2 - jet_py**2 - jet_pz**2
    jet_mass2 = np.maximum(jet_mass2, 0)
    jet_mass = np.sqrt(jet_mass2)

    jet_y = np.divide((pt * y).sum(axis=1), jet_pt, out=np.zeros_like(jet_pt), where=jet_pt > 0)
    jet_phi_x = (pt * np.cos(phi)).sum(axis=1)
    jet_phi_y = (pt * np.sin(phi)).sum(axis=1)
    jet_phi = np.arctan2(jet_phi_y, jet_phi_x)

    dy = y - jet_y[:, None]
    dphi = safe_delta_phi(phi, jet_phi[:, None])
    dr = np.sqrt(dy**2 + dphi**2)

    girth = np.divide((pt * dr).sum(axis=1), jet_pt, out=np.zeros_like(jet_pt), where=jet_pt > 0)
    pTD = np.divide(np.sqrt((pt**2).sum(axis=1)), jet_pt, out=np.zeros_like(jet_pt), where=jet_pt > 0)

    frac_r_0_1 = np.divide((pt * (dr < 0.1)).sum(axis=1), jet_pt, out=np.zeros_like(jet_pt), where=jet_pt > 0)
    frac_r_0_2 = np.divide((pt * (dr < 0.2)).sum(axis=1), jet_pt, out=np.zeros_like(jet_pt), where=jet_pt > 0)
    frac_r_0_3 = np.divide((pt * (dr < 0.3)).sum(axis=1), jet_pt, out=np.zeros_like(jet_pt), where=jet_pt > 0)
    frac_r_0_4 = np.divide((pt * (dr < 0.4)).sum(axis=1), jet_pt, out=np.zeros_like(jet_pt), where=jet_pt > 0)

    pt_masked = np.where(mask, pt, np.nan)
    mean_const_pt = np.nanmean(pt_masked, axis=1)
    std_const_pt  = np.nanstd(pt_masked, axis=1)
    max_const_pt_frac = np.divide(
        np.nanmax(pt_masked, axis=1),
        jet_pt,
        out=np.zeros_like(jet_pt),
        where=jet_pt > 0
    )

    df = pd.DataFrame({
        "jet_pt": jet_pt,
        "jet_mass": jet_mass,
        "n_particles": n_particles,
        "girth": girth,
        "pTD": pTD,
        "frac_r_0_1": frac_r_0_1,
        "frac_r_0_2": frac_r_0_2,
        "frac_r_0_3": frac_r_0_3,
        "frac_r_0_4": frac_r_0_4,
        "mean_const_pt": mean_const_pt,
        "std_const_pt": std_const_pt,
        "max_const_pt_frac": max_const_pt_frac,
    })

    return df

In [ ]:
features = compute_jet_features(X)
features["label"] = y
features.head()

## 6. Plot quark vs gluon feature distributions

In [ ]:
plot_features = ["jet_mass", "jet_pt", "n_particles", "girth", "pTD", "max_const_pt_frac"]

fig, axes = plt.subplots(2, 3, figsize=(20, 10))
axes = axes.flatten()

for ax, feat in zip(axes, plot_features):
    sns.histplot(features[features["label"] == 1][feat], bins=50, stat="density",
                 alpha=0.5, label="Quark", ax=ax)
    sns.histplot(features[features["label"] == 0][feat], bins=50, stat="density",
                 alpha=0.5, label="Gluon", ax=ax)
    ax.set_title(feat)
    ax.legend()

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/feature_distributions.png", dpi=200, bbox_inches="tight")
plt.show()

## 7. Train / validation / test split

In [ ]:
X_tab = features.drop(columns=["label"])
y_tab = features["label"].values

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_tab, y_tab, test_size=0.15, stratify=y_tab, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.1765, stratify=y_train_full, random_state=42
)

print(X_train.shape, X_val.shape, X_test.shape)

## 8. Standardize tabular features

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

## 9. Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=14,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train, y_train)

rf_test_proba = rf.predict_proba(X_test)[:, 1]
rf_test_pred = (rf_test_proba >= 0.5).astype(int)

rf_acc = accuracy_score(y_test, rf_test_pred)
rf_auc = roc_auc_score(y_test, rf_test_proba)

print("Random Forest Accuracy:", rf_acc)
print("Random Forest AUC:", rf_auc)
print(classification_report(y_test, rf_test_pred))

## 10. Random Forest feature importance

In [ ]:
rf_importance = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=rf_importance.values, y=rf_importance.index)
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/rf_feature_importance.png", dpi=200, bbox_inches="tight")
plt.show()

## 11. XGBoost

In [ ]:
xgb = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

xgb_test_proba = xgb.predict_proba(X_test)[:, 1]
xgb_test_pred = (xgb_test_proba >= 0.5).astype(int)

xgb_acc = accuracy_score(y_test, xgb_test_pred)
xgb_auc = roc_auc_score(y_test, xgb_test_proba)

print("XGBoost Accuracy:", xgb_acc)
print("XGBoost AUC:", xgb_auc)
print(classification_report(y_test, xgb_test_pred))

## 12. XGBoost feature importance

In [ ]:
xgb_importance = pd.Series(xgb.feature_importances_, index=X_train.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=xgb_importance.values, y=xgb_importance.index)
plt.title("XGBoost Feature Importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/xgb_feature_importance.png", dpi=200, bbox_inches="tight")
plt.show()

## 13. Confusion matrix helper

In [ ]:
def plot_cm(y_true, y_pred, title, save_name):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Gluon", "Quark"],
                yticklabels=["Gluon", "Quark"])
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/{save_name}", dpi=200, bbox_inches="tight")
    plt.show()

In [ ]:
plot_cm(y_test, rf_test_pred, "Random Forest Confusion Matrix", "rf_confusion_matrix.png")
plot_cm(y_test, xgb_test_pred, "XGBoost Confusion Matrix", "xgb_confusion_matrix.png")

## 14. Convert jets to images

In [ ]:
def jets_to_images(X, image_size=33, y_max=0.8, phi_max=0.8):
    pt  = X[:, :, 0]
    y   = X[:, :, 1]
    phi = X[:, :, 2]

    jet_pt = pt.sum(axis=1)
    jet_y = np.divide((pt * y).sum(axis=1), jet_pt, out=np.zeros_like(jet_pt), where=jet_pt > 0)
    jet_phi_x = (pt * np.cos(phi)).sum(axis=1)
    jet_phi_y = (pt * np.sin(phi)).sum(axis=1)
    jet_phi = np.arctan2(jet_phi_y, jet_phi_x)

    images = np.zeros((X.shape[0], image_size, image_size), dtype=np.float32)

    y_edges = np.linspace(-y_max, y_max, image_size + 1)
    phi_edges = np.linspace(-phi_max, phi_max, image_size + 1)

    for i in range(X.shape[0]):
        valid = pt[i] > 0
        if not np.any(valid):
            continue

        dy = y[i, valid] - jet_y[i]
        dphi = safe_delta_phi(phi[i, valid], jet_phi[i])

        w = pt[i, valid]
        hist, _, _ = np.histogram2d(dy, dphi, bins=[y_edges, phi_edges], weights=w)

        if hist.sum() > 0:
            hist = hist / hist.sum()

        images[i] = hist

    return images[..., None]

In [ ]:
n_cnn = 40000
X_cnn = X[:n_cnn]
y_cnn = y[:n_cnn]

images = jets_to_images(X_cnn, image_size=33)
print("Jet images shape:", images.shape)

## 15. Average quark and gluon jet images

In [ ]:
avg_quark = images[y_cnn == 1].mean(axis=0).squeeze()
avg_gluon = images[y_cnn == 0].mean(axis=0).squeeze()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im0 = axes[0].imshow(avg_quark, origin="lower", cmap="inferno")
axes[0].set_title("Average Quark Jet Image")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(avg_gluon, origin="lower", cmap="inferno")
axes[1].set_title("Average Gluon Jet Image")
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/average_jet_images.png", dpi=200, bbox_inches="tight")
plt.show()

## 16. CNN data split

In [ ]:
X_train_img, X_test_img, y_train_img, y_test_img = train_test_split(
    images, y_cnn, test_size=0.15, stratify=y_cnn, random_state=42
)

X_train_img, X_val_img, y_train_img, y_val_img = train_test_split(
    X_train_img, y_train_img, test_size=0.1765, stratify=y_train_img, random_state=42
)

print(X_train_img.shape, X_val_img.shape, X_test_img.shape)

## 17. CNN model

In [ ]:
cnn = models.Sequential([
    layers.Input(shape=X_train_img.shape[1:]),
    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid")
])

cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)

cnn.summary()

## 18. Train CNN

In [ ]:
early_stop = EarlyStopping(
    monitor="val_auc",
    mode="max",
    patience=5,
    restore_best_weights=True
)

history = cnn.fit(
    X_train_img, y_train_img,
    validation_data=(X_val_img, y_val_img),
    epochs=25,
    batch_size=128,
    callbacks=[early_stop],
    verbose=1
)

## 19. Plot CNN training curves

In [ ]:
history_df = pd.DataFrame(history.history)

plt.figure(figsize=(10, 5))
plt.plot(history_df["accuracy"], label="Train Accuracy")
plt.plot(history_df["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("CNN Training Accuracy")
plt.legend()
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/cnn_accuracy_curve.png", dpi=200, bbox_inches="tight")
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(history_df["auc"], label="Train AUC")
plt.plot(history_df["val_auc"], label="Validation AUC")
plt.xlabel("Epoch")
plt.ylabel("AUC")
plt.title("CNN Training AUC")
plt.legend()
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/cnn_auc_curve.png", dpi=200, bbox_inches="tight")
plt.show()

## 20. Evaluate CNN

In [ ]:
cnn_test_proba = cnn.predict(X_test_img).ravel()
cnn_test_pred = (cnn_test_proba >= 0.5).astype(int)

cnn_acc = accuracy_score(y_test_img, cnn_test_pred)
cnn_auc = roc_auc_score(y_test_img, cnn_test_proba)

print("CNN Accuracy:", cnn_acc)
print("CNN AUC:", cnn_auc)
print(classification_report(y_test_img, cnn_test_pred))

In [ ]:
plot_cm(y_test_img, cnn_test_pred, "CNN Confusion Matrix", "cnn_confusion_matrix.png")

## 21. ROC comparison

In [ ]:
rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_test_proba)
xgb_fpr, xgb_tpr, _ = roc_curve(y_test, xgb_test_proba)
cnn_fpr, cnn_tpr, _ = roc_curve(y_test_img, cnn_test_proba)

plt.figure(figsize=(10, 8))
plt.plot(rf_fpr, rf_tpr, label=f"Random Forest (AUC = {rf_auc:.4f})")
plt.plot(xgb_fpr, xgb_tpr, label=f"XGBoost (AUC = {xgb_auc:.4f})")
plt.plot(cnn_fpr, cnn_tpr, label=f"CNN (AUC = {cnn_auc:.4f})")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/roc_comparison.png", dpi=200, bbox_inches="tight")
plt.show()

## 22. Permutation importance for XGBoost

In [ ]:
perm = permutation_importance(
    xgb, X_test, y_test,
    n_repeats=5,
    random_state=42,
    scoring="roc_auc"
)

perm_importance = pd.Series(
    perm.importances_mean,
    index=X_test.columns
).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=perm_importance.values, y=perm_importance.index)
plt.title("Permutation Importance (XGBoost)")
plt.xlabel("Mean Importance")
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/xgb_permutation_importance.png", dpi=200, bbox_inches="tight")
plt.show()

## 23. Final results summary

In [ ]:
results = pd.DataFrame({
    "Model": ["Random Forest", "XGBoost", "CNN Jet Images"],
    "Accuracy": [rf_acc, xgb_acc, cnn_acc],
    "AUC": [rf_auc, xgb_auc, cnn_auc]
})

results = results.sort_values("AUC", ascending=False)
print(results)

results.to_csv(f"{SAVE_DIR}/results_summary.csv", index=False)

## 24. Save a quick text report

In [ ]:
with open(f"{SAVE_DIR}/summary_report.txt", "w") as f:
    f.write("Quark vs Gluon Jet Classification Results\n")
    f.write("=" * 50 + "\n\n")
    f.write(results.to_string(index=False))
    f.write("\n\n")
    f.write("Random Forest Classification Report\n")
    f.write(classification_report(y_test, rf_test_pred))
    f.write("\n\n")
    f.write("XGBoost Classification Report\n")
    f.write(classification_report(y_test, xgb_test_pred))
    f.write("\n\n")
    f.write("CNN Classification Report\n")
    f.write(classification_report(y_test_img, cnn_test_pred))

print("Saved report files in", SAVE_DIR)

## 25. Expected outputs

After running this notebook, the `qg_results/` folder should contain:

- feature distribution plots
- feature importance plots
- confusion matrices
- average jet images
- CNN training curves
- ROC comparison
- results summary CSV
- text report